In [1]:
import os
import sys
import numpy as np
import polars as pl
from mars.feature.selector import MarsStatsSelector
from mars.utils.logger import set_log_level

# 1. 环境准备
sys.path.append(os.getcwd())
set_log_level("WARNING") 

# ==============================================================================
# 2. 构造模拟数据集 (Mock Data)
# ==============================================================================
def create_mock_data(n=10000):
    np.random.seed(42)
    
    # 基础列
    target = np.random.choice([0, 1], size=n, p=[0.9, 0.1])
    months = np.random.choice(["2023-01", "2023-02", "2023-03"], size=n)
    
    # --- 特征集构造 ---
    # A. 强特征系列
    f_strong = np.where(target == 1, np.random.normal(5, 2, n), np.random.normal(2, 2, n))
    f_strong_corr = f_strong * 2.5 + np.random.normal(0, 0.1, n)
    
    # B. 质量问题系列 (缺失、单一值、高0值)
    f_high_missing = np.random.normal(0, 1, n)
    f_high_missing[np.random.rand(n) < 0.96] = -999 # 模拟自定义缺失
    
    f_single_value = np.ones(n)
    f_single_value[0:50] = 0 # 只有 0.5% 的非众数值
    
    # [新增测试点] 高0值率特征：98% 的数据都是 0
    f_high_zero = np.random.normal(5, 2, n)
    f_high_zero[np.random.rand(n) < 0.98] = 0 
    
    # C. 业务特殊值与类别
    f_special = np.where(target == 1, np.random.normal(5, 2, n), np.random.normal(2, 2, n))
    f_special[np.random.rand(n) < 0.15] = -998 # 15% 特殊值
    
    f_category = np.where(target == 1, 
                          np.random.choice(["A", "B", "C"], size=n, p=[0.6, 0.3, 0.1]),
                          np.random.choice(["A", "B", "C"], size=n, p=[0.2, 0.5, 0.3]))
    
    # D. 稳定性与噪音
    f_noise = np.random.normal(0, 1, n)
    
    f_unstable = np.where(target == 1, np.random.normal(5, 2, n), np.random.normal(2, 2, n))
    f_unstable[months == "2023-03"] = np.random.normal(10, 5, sum(months == "2023-03"))
    
    f_logic_flip = np.zeros(n)
    m12, m3 = (months != "2023-03"), (months == "2023-03")
    f_logic_flip[m12] = np.where(target[m12]==1, np.random.normal(5, 2, sum(m12)), np.random.normal(2, 2, sum(m12)))
    f_logic_flip[m3] = np.where(target[m3]==1, np.random.normal(2, 2, sum(m3)), np.random.normal(5, 2, sum(m3)))

    return pl.DataFrame({
        "target": target, "month": months,
        "f_strong": f_strong, "f_strong_corr": f_strong_corr,
        "f_high_missing": f_high_missing, "f_single_value": f_single_value,
        "f_high_zero": f_high_zero,  # 新增列
        "f_special": f_special, "f_category": f_category,
        "f_noise": f_noise, "f_unstable": f_unstable, 
        "f_logic_flip": f_logic_flip, "f_white": f_noise, "f_black": f_strong, "f_ignored": f_noise
    })

df_mock = create_mock_data()
print(f"📊 数据生成完毕: {df_mock.shape}")

# ==============================================================================
# 3. 初始化筛选器 (参数配置)
# ==============================================================================
# 包含新特征的评估池
feature_pool = [
    "f_strong", "f_strong_corr", "f_high_missing", "f_single_value",
    "f_high_zero", # 新增到特征池
    "f_special", "f_category", "f_noise", "f_unstable", 
    "f_logic_flip", "f_white", "f_black"
]

selector = MarsStatsSelector(
    target="target",
    features=feature_pool,
    profile_by="month",
    
    # 业务定义
    white_list=["f_white"],
    black_list=["f_black"],
    missing_values=[-999],
    special_values=[-998],
    
    # --- Stage 1: 数据质量 (包含 0 值率新参数) ---
    missing_thr=0.95,
    mode_thr=0.99,
    zeros_thr=0.95,        # [新增测试配置] 0 值率超过 95% 则杀掉
    
    # --- Stage 2 & 3: 粗筛与精筛 ---
    rough_iv_thr=0.01,
    iv_thr=0.02,
    lift_thr=2.0,    # 开启精筛 Lift 召回
    min_sample_rate=0.05,
    
    # --- Stage 4, 5, 6: 稳定性与冗余 ---
    psi_thr=0.25,
    rc_thr=0.5,
    corr_thr=0.8
)

# ==============================================================================
# 4. 执行筛选流程
# ==============================================================================
print("\n🚀 正在执行特征筛选大逃杀...")
selector.fit(df_mock)

# 转换数据 (只保留幸存者)
df_selected = selector.transform(df_mock)

# ==============================================================================
# 5. 结果展示与闭环功能验证
# ==============================================================================
print(f"\n✅ 筛选完成! 幸存特征数量: {len(selector.selected_features_)}")

# A. 打印尸检报告 (用于验证 f_high_zero 是否死于 "High Zero Rate")
print("\n📋 筛选过程溯源 (尸检报告):")
report_df = selector.get_report()

try:
    from IPython.display import display
    # 让 Selected 变绿，Dropped 变红，方便肉眼快速审计
    display(report_df.to_pandas().style.map(
        lambda x: 'color: #27ae60; font-weight:bold' if x == 'Selected' else 'color: #e74c3c',
        subset=['status']
    ))
except:
    print(report_df)

# B. 获取详细业务评估报告 (Excel 级别报表)
print("\n📊 正在生成最终入选特征的业务明细报表...")
eval_report = selector.get_eval_report(df_mock)
# eval_report.show_summary() # 如果在 Jupyter 中请取消注释直接查看

# C. 保存筛选名单经验 (用于下一次循环增量运行)
# print("\n💾 正在保存筛选名单 (模糊匹配 'quality' 和 'scan' 阶段的垃圾特征) ...")
# selector.save_selector_lists(
#     path="mars_knowledge_base.json", 
#     blacklist_stages=['quality', 'scan'] # 把质量差和IV低的永久拉黑
# )

# print("\n✨ 测试流程全部结束。")

📊 数据生成完毕: (10000, 15)

🚀 正在执行特征筛选大逃杀...


Quality Check:   0%|          | 0/11 [00:00<?, ?it/s]

PSI Check:   0%|          | 0/7 [00:00<?, ?it/s]

RC Check:   0%|          | 0/5 [00:00<?, ?it/s]

Corr Check:   0%|          | 0/5 [00:00<?, ?it/s]


✅ 筛选完成! 幸存特征数量: 4

📋 筛选过程溯源 (尸检报告):


,feature,status,stage,reason,value,description
0,f_strong_corr,Dropped,Corr_Filter,Correlated with 'f_strong',0.992777,
1,f_noise,Dropped,Fine_Scan,Low IV & No Lift Recall,0.007796,
2,f_black,Dropped,Init,Black List,-1.000000,
3,f_high_missing,Dropped,Quality,High Missing,0.963700,
4,f_high_zero,Dropped,Quality,High Zero Rate,0.981000,
5,f_single_value,Dropped,Quality,Single Value (Mode),0.995000,
6,f_logic_flip,Dropped,Stability,High PSI (1.09),1.090734,
7,f_unstable,Dropped,Stability,High PSI (2.64),2.638993,
8,f_category,Selected,Corr_Filter,Independent,0.721761,
9,f_special,Selected,Corr_Filter,Independent,1.777195,



📊 正在生成最终入选特征的业务明细报表...


In [2]:
eval_report

In [3]:
eval_report.show_summary()

feature,iv,ks,auc,psi_max,rc_min,mono
f_strong,2.1396,55.3175,0.8545,0.0068,0.9724,1.0000
f_special,1.7772,45.0458,0.8176,0.0079,0.9948,1.0000
f_category,0.7218,38.8613,0.7141,0.0037,0.9982,1.0000
f_white,0.0078,2.0373,0.5176,0.0028,0.1987,-1.0000
